# Level 1 Limit Order Book Simulation

This notebook demonstrates the **Level 1 LOB Simulator** implemented in `order_book_simulator.py`.

## Model Overview

We track only the **best bid** $(p_b, q_b)$ and **best ask** $(p_a, q_a)$.  
Six competing Poisson processes drive the dynamics:

| Process | Rate | Effect |
|---------|------|--------|
| Limit sell at ask | $\lambda_a$ | $q_a \mathrel{+}= 1$ |
| Limit buy  at bid | $\lambda_b$ | $q_b \mathrel{+}= 1$ |
| Market buy  (hits ask) | $\lambda_{ma}$ | $q_a \mathrel{-}= 1$; if $q_a{=}0$ then $p_a \mathrel{+}= \delta$ |
| Market sell (hits bid) | $\lambda_{mb}$ | $q_b \mathrel{-}= 1$; if $q_b{=}0$ then $p_b \mathrel{-}= \delta$ |
| Cancel at ask | $\theta_a \cdot q_a$ | $q_a \mathrel{-}= 1$; if $q_a{=}0$ then $p_a \mathrel{+}= \delta$ |
| Cancel at bid | $\theta_b \cdot q_b$ | $q_b \mathrel{-}= 1$; if $q_b{=}0$ then $p_b \mathrel{-}= \delta$ |

The **imbalance** at any point is
$$I = \frac{q_b}{q_a + q_b}$$
and the **Stoikov micro-price** is
$$\tilde{p} = I \cdot p_a + (1-I) \cdot p_b$$

This is the same imbalance computed from real MBO data in the `SR1.ipynb` and `ZBU2.ipynb` notebooks.

**Simulation method:** Gillespie direct algorithm — exact continuous-time Monte Carlo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from order_book_simulator import LOBSimulator, LOBAnalyzer, LOBPlotter, EventType

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.5f}".format)

# Tick size matching the SR1J2 instrument used in the rest of this project
TICK = 0.005

## 1. Baseline Simulation — Symmetric Flow

Equal limit-order and market-order intensities on both sides.  
Expect the mid-price to random-walk with no drift, and imbalance to hover near 0.5.

In [ ]:
sim = LOBSimulator(
    pa=100.005, qa=10,
    pb=100.000, qb=10,
    lambda_a  = 5.0,   # limit sell arrivals
    lambda_b  = 5.0,   # limit buy  arrivals
    lambda_ma = 2.0,   # market buy  arrivals
    lambda_mb = 2.0,   # market sell arrivals
    theta_a   = 0.5,   # cancellation rate per unit at ask
    theta_b   = 0.5,   # cancellation rate per unit at bid
    tick_size = TICK,
    seed      = 42,
)

history = sim.run(n_events=5_000)
print(f"Simulated {len(history)-1} events  |  "
      f"Simulated time = {history['time'].iloc[-1]:.4f}  |  "
      f"Rows in DataFrame: {len(history)}")
history.head(8)

In [ ]:
analyzer = LOBAnalyzer(history)
analyzer.summary_stats()

In [ ]:
print("Event frequency:")
analyzer.event_frequency()

In [ ]:
plotter = LOBPlotter(history, tick_size=TICK)
fig, axes = plotter.plot_dashboard()
fig.suptitle("Level 1 LOB Simulation — Symmetric Flow (λ_a=λ_b=5, λ_ma=λ_mb=2)",
             fontsize=12, y=1.01)
plt.show()

In [ ]:
# Event timeline: visual check that all six processes are firing
fig, ax = plotter.plot_event_timeline(max_events=300)
plt.tight_layout()
plt.show()

## 2. Asymmetric Flow — Buy Pressure

Increase market-buy intensity and limit-buy intensity relative to the sell side.  
**Prediction:** ask and bid prices drift upward; imbalance distribution shifts above 0.5.

In [ ]:
sim_buy = LOBSimulator(
    pa=100.005, qa=10,
    pb=100.000, qb=10,
    lambda_a  = 3.0,   # fewer limit sells — thinner ask
    lambda_b  = 7.0,   # more   limit buys  — thicker bid
    lambda_ma = 4.0,   # aggressive market buys
    lambda_mb = 1.0,   # few     market sells
    theta_a   = 0.2,
    theta_b   = 0.8,
    tick_size = TICK,
    seed      = 99,
)

hist_buy = sim_buy.run(n_events=5_000)
LOBAnalyzer(hist_buy).summary_stats()

In [ ]:
fig, axes = LOBPlotter(hist_buy, TICK).plot_dashboard()
fig.suptitle("Level 1 LOB — Buy Pressure (λ_ma=4, λ_mb=1)", fontsize=12, y=1.01)
plt.show()

## 3. Comparing Imbalance Distributions

Symmetric flow → imbalance centred at 0.5.  
Buy pressure → imbalance skewed toward 1 (bid-heavy queue).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(history["imbalance"],     bins=40, alpha=0.55, color="steelblue",
        label="Symmetric",        density=True)
ax.hist(hist_buy["imbalance"],    bins=40, alpha=0.55, color="crimson",
        label="Buy pressure",     density=True)
ax.axvline(0.5, color="black", ls="--", lw=1.0)
ax.set_xlabel(r"Imbalance  $I = Q_b\,/\,(Q_a + Q_b)$", fontsize=11)
ax.set_ylabel("Density")
ax.set_title("Simulated Imbalance Distribution")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Micro-price vs Mid-price

The **Stoikov micro-price** $\tilde{p} = I \cdot p_a + (1-I)\cdot p_b$  
anticipates direction: when the bid queue is larger ($I > 0.5$), $\tilde{p}$ sits above mid, reflecting upward pressure.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)

for ax, h, title in [
    (axes[0], history,  "Symmetric"),
    (axes[1], hist_buy, "Buy Pressure"),
]:
    ax.step(h["time"], h["mid"],         where="post",
            color="black", lw=1.0, label="Mid-price",   alpha=0.8)
    ax.step(h["time"], h["micro_price"], where="post",
            color="royalblue", lw=1.0, ls="--", label="Micro-price", alpha=0.85)
    ax.set_xlabel("Time")
    ax.set_ylabel("Price")
    ax.set_title(f"Micro-price vs Mid-price — {title}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

## 5. Parameter Sensitivity — Cancellation Rates

Higher cancellation rates thin the queue faster, increasing tick frequency and spread volatility.

In [ ]:
results = []
for theta in [0.0, 0.25, 0.5, 1.0, 2.0]:
    s = LOBSimulator(
        pa=100.005, qa=10, pb=100.000, qb=10,
        lambda_a=5, lambda_b=5, lambda_ma=2, lambda_mb=2,
        theta_a=theta, theta_b=theta,
        tick_size=TICK, seed=0,
    )
    h = s.run(n_events=3_000)
    row = LOBAnalyzer(h).summary_stats().iloc[0].to_dict()
    row["theta"] = theta
    results.append(row)

sens = pd.DataFrame(results)[
    ["theta", "mean_spread", "std_spread",
     "mean_qa", "mean_qb", "ask_price_ups", "bid_price_downs"]
]
sens

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(sens["theta"].astype(str), sens["mean_spread"], color="purple", alpha=0.7)
axes[0].set_xlabel("θ (cancellation rate)")
axes[0].set_ylabel("Mean spread")
axes[0].set_title("Mean Spread vs θ")
axes[0].grid(True, alpha=0.3)

axes[1].bar(sens["theta"].astype(str), sens["mean_qa"],
            color="red",  alpha=0.7, label="Ask qty")
axes[1].bar(sens["theta"].astype(str), sens["mean_qb"],
            color="green", alpha=0.5, label="Bid qty")
axes[1].set_xlabel("θ (cancellation rate)")
axes[1].set_ylabel("Mean queue depth")
axes[1].set_title("Queue Depth vs θ")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].bar(sens["theta"].astype(str), sens["ask_price_ups"],
            color="red",  alpha=0.7, label="Ask tick-ups")
axes[2].bar(sens["theta"].astype(str), sens["bid_price_downs"],
            color="blue", alpha=0.5, label="Bid tick-downs")
axes[2].set_xlabel("θ (cancellation rate)")
axes[2].set_ylabel("# price moves")
axes[2].set_title("Price Move Frequency vs θ")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Multiple Paths — Monte Carlo

Run $N$ independent paths to estimate the distribution of the mid-price at horizon $T$.

In [ ]:
N_PATHS = 200
N_EVENTS = 2_000
final_mids = []

base_sim = LOBSimulator(
    pa=100.005, qa=10, pb=100.000, qb=10,
    lambda_a=5, lambda_b=5, lambda_ma=2, lambda_mb=2,
    theta_a=0.5, theta_b=0.5,
    tick_size=TICK,
)

for seed in range(N_PATHS):
    base_sim.reset(seed=seed)
    h = base_sim.run(n_events=N_EVENTS)
    final_mids.append(h["mid"].iloc[-1])

final_mids = np.array(final_mids)
print(f"Mid-price after {N_EVENTS} events:")
print(f"  Mean  = {final_mids.mean():.5f}")
print(f"  Std   = {final_mids.std():.5f}")
print(f"  Min   = {final_mids.min():.5f}")
print(f"  Max   = {final_mids.max():.5f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(final_mids, bins=30, color="steelblue", alpha=0.75, edgecolor="white")
ax.axvline(100.0025, color="black", ls="--", lw=1.2, label=f"Initial mid = {100.0025}")
ax.axvline(final_mids.mean(), color="red", ls="-", lw=1.2,
           label=f"Simulated mean = {final_mids.mean():.5f}")
ax.set_xlabel("Final mid-price")
ax.set_ylabel("Count")
ax.set_title(f"Distribution of Final Mid-price ({N_PATHS} paths, {N_EVENTS} events each)")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Connection to Real Data

The intensity parameters $\lambda_a, \lambda_b, \lambda_{ma}, \lambda_{mb}, \theta_a, \theta_b$  
can be estimated from the SR1J2 MBO data already in this project:

| Parameter | Empirical estimator |
|-----------|---------------------|
| $\lambda_{ma}$ | Market buy count / total time window |
| $\lambda_{mb}$ | Market sell count / total time window |
| $\lambda_a$ | Limit sell count at best ask / time |
| $\lambda_b$ | Limit buy count at best bid / time |
| $\theta_a$ | Cancel-at-ask count / (time × mean ask depth) |
| $\theta_b$ | Cancel-at-bid count / (time × mean bid depth) |

The imbalance formula $I = q_b\,/\,(q_a+q_b)$ is identical to the `imb` column  
computed in `SR1.ipynb`, closing the loop between the simulator and real microstructure data.